# Ingest WSR-88D Level-II data with GV Tools 0.65.0

This notebook discovers WSR-88D Level-II files below a user-selected input directory and reads a configurable number of volumes. It does not write generated output inside the package directory.

## Installation (`tcsh`)

```tcsh
set WORKSPACE = /Users/dwolff/Desktop/Work/GV_Tools
set PACKAGE_DIR = $WORKSPACE/gv_tools
set PYTHON = $HOME/anaconda3/bin/python
$PYTHON -m pip install --upgrade "$PACKAGE_DIR/dist/gv_tools-0.65.0-py3-none-any.whl[radar,notebook]"
$PYTHON -m jupyter lab $PACKAGE_DIR/notebooks/Ingest_WSR88D.ipynb
```

In [ ]:
from datetime import date, timedelta
from pathlib import Path

import gv_tools

assert gv_tools.__version__ == "0.65.0", gv_tools.__version__
print(f"gv_tools version: {gv_tools.__version__}")

## User configuration

Set `INPUT_ROOT` to the directory above the date/site folders. `LOAD_LIMIT` prevents an accidental multi-day in-memory load; set it to `None` to read every discovered volume.

In [ ]:
WORKSPACE = Path("/Users/dwolff/Desktop/Work/GV_Tools")
PACKAGE_DIRECTORY = WORKSPACE / "gv_tools"
OUTPUT_DIRECTORY = WORKSPACE / "Output"

SITE = "KCRP"
START_DATE = date(2024, 8, 21)
END_DATE = date(2024, 8, 23)
INPUT_ROOT = Path("/Users/dwolff/Desktop/TS_Harold_DSD/KCRP_Level2_20240821_20240823")
LOAD_LIMIT = 1

assert OUTPUT_DIRECTORY.resolve() != PACKAGE_DIRECTORY.resolve()
print(f"Input:  {INPUT_ROOT}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
files = []
day = START_DATE
while day <= END_DATE:
    day_directory = INPUT_ROOT / f"{day:%Y}" / f"{day:%m}" / f"{day:%d}" / SITE
    files.extend(sorted(day_directory.glob("*_V06")))
    day += timedelta(days=1)

if not files:
    raise FileNotFoundError(f"No {SITE} Level-II *_V06 files found below {INPUT_ROOT}")

selected_files = files if LOAD_LIMIT is None else files[:LOAD_LIMIT]
print(f"Discovered {len(files)} volumes; loading {len(selected_files)}")
radars = [gv_tools.io.read_radar(path) for path in selected_files]

In [ ]:
radars[0].info()